# ProvenanceLens Phase 2 Prototype

**Team:** Neal Salian (I062), Rusheel Sharma (I069), Veer Shetty (I071)

This notebook demonstrates a partial evidence-calibrated workflow for direct model-lineage metadata repair. It uses frozen sample evidence so results are reproducible. The next stage will connect the same pipeline to selected Hugging Face repositories.

## Current architecture

`Evidence bundle -> deterministic structured parsing -> prose claim extraction -> evidence scoring -> KEEP / ADD / REPLACE / ABSTAIN`

The prototype preserves abstention when evidence conflicts or when a retrieval tool fails.

In [1]:
# Run this once in Google Colab if the packages are unavailable.
# !pip install -q langchain-core pydantic huggingface_hub

import json
import re
from collections import defaultdict
from typing import Any, Dict, List, Literal, Optional

try:
    from pydantic import BaseModel, Field
    from langchain_core.prompts import PromptTemplate
    from langchain_core.output_parsers import JsonOutputParser
    from langchain_core.runnables import RunnableLambda
    LANGCHAIN_AVAILABLE = True
except ImportError:
    LANGCHAIN_AVAILABLE = False

print("LangChain available:", LANGCHAIN_AVAILABLE)


LangChain available: True


In [2]:
if LANGCHAIN_AVAILABLE:
    class LineageDecision(BaseModel):
        model_id: str
        action: Literal["KEEP", "ADD", "REPLACE", "ABSTAIN"]
        proposed_parent: Optional[str] = None
        relationship: Optional[str] = None
        confidence: float = Field(ge=0.0, le=1.0)
        supporting_evidence: List[str]
        conflicting_evidence: List[str]
        rationale: str

    claim_prompt = PromptTemplate.from_template(
        "Extract only explicit direct-parent model claims from the text below. "
        "Return JSON with parent_model, relationship, quote and source. "
        "Do not guess.\n\nSOURCE: {source}\nTEXT: {text}"
    )
    decision_parser = JsonOutputParser(pydantic_object=LineageDecision)

RELATION_PATTERNS = {
    "fine-tune": r"(?:fine[- ]?tuned|trained)\s+(?:from|on top of)\s+([\w.-]+/[\w.-]+)",
    "adapter": r"adapter\s+(?:for|based on)\s+([\w.-]+/[\w.-]+)",
    "merge": r"merg(?:e|ed)\s+(?:from|of)\s+([\w.-]+/[\w.-]+)",
    "quantized": r"quantiz(?:ed|ation)\s+(?:from|of)\s+([\w.-]+/[\w.-]+)",
}


## Frozen evidence bundles

The demo cases are synthetic and designed to test all four actions plus tool-failure handling. They are not claims about real repositories.

In [3]:
SAMPLES = [
    {
        "model_id": "demo/missing-lineage-model",
        "metadata": {"base_model": None, "base_model_relation": None},
        "config": {"_name_or_path": "meta-llama/Llama-3.2-1B"},
        "readme": "This model was fine-tuned from meta-llama/Llama-3.2-1B for instruction following.",
        "tool_errors": [],
    },
    {
        "model_id": "demo/conflicting-lineage-model",
        "metadata": {"base_model": "mistralai/Mistral-7B-v0.1", "base_model_relation": "fine-tune"},
        "config": {"_name_or_path": "meta-llama/Llama-3.2-1B"},
        "readme": "This model was fine-tuned from meta-llama/Llama-3.2-1B.",
        "tool_errors": [],
    },
    {
        "model_id": "demo/correct-lineage-model",
        "metadata": {"base_model": "meta-llama/Llama-3.2-1B", "base_model_relation": "fine-tune"},
        "config": {"_name_or_path": "meta-llama/Llama-3.2-1B"},
        "readme": "This model was fine-tuned from meta-llama/Llama-3.2-1B.",
        "tool_errors": [],
    },
    {
        "model_id": "demo/ambiguous-lineage-model",
        "metadata": {"base_model": None, "base_model_relation": None},
        "config": {"_name_or_path": "Qwen/Qwen2.5-1.5B"},
        "readme": "This model was fine-tuned from meta-llama/Llama-3.2-1B.",
        "tool_errors": [],
    },
    {
        "model_id": "demo/tool-failure-model",
        "metadata": {"base_model": None, "base_model_relation": None},
        "config": {},
        "readme": "",
        "tool_errors": ["README retrieval timed out", "config.json unavailable"],
    },
]


In [4]:
def extract_evidence(bundle: Dict[str, Any]) -> Dict[str, Any]:
    claims = []
    declared = bundle.get("metadata", {}).get("base_model")
    relation = bundle.get("metadata", {}).get("base_model_relation")
    if declared:
        claims.append({"parent": declared, "relation": relation, "source": "metadata", "weight": 0})

    config_parent = bundle.get("config", {}).get("_name_or_path")
    if config_parent and "/" in config_parent:
        claims.append({"parent": config_parent, "relation": relation or "fine-tune", "source": "config.json", "weight": 3})

    text = bundle.get("readme", "")
    for rel, pattern in RELATION_PATTERNS.items():
        match = re.search(pattern, text, flags=re.I)
        if match:
            claims.append({"parent": match.group(1).rstrip("."), "relation": rel, "source": "README", "weight": 2})
    return {**bundle, "claims": claims}


def decide_lineage(item: Dict[str, Any]) -> Dict[str, Any]:
    declared = item.get("metadata", {}).get("base_model")
    errors = item.get("tool_errors", [])
    independent = [c for c in item["claims"] if c["source"] != "metadata"]
    scores = defaultdict(int)
    sources = defaultdict(list)
    relations = defaultdict(list)
    for claim in independent:
        scores[claim["parent"]] += claim["weight"]
        sources[claim["parent"]].append(claim["source"])
        relations[claim["parent"]].append(claim["relation"])

    if not scores:
        return {
            "model_id": item["model_id"], "action": "ABSTAIN", "proposed_parent": None,
            "relationship": None, "confidence": 0.10, "supporting_evidence": [],
            "conflicting_evidence": errors or ["No independent parent evidence found"],
            "rationale": "Evidence is unavailable or insufficient for a safe repair."
        }

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    winner, top_score = ranked[0]
    runner_score = ranked[1][1] if len(ranked) > 1 else 0
    conflict = len(ranked) > 1
    evidence_count = len(set(sources[winner]))

    if conflict and top_score - runner_score < 2:
        action = "ABSTAIN"
        confidence = 0.45
        parent = None
    elif evidence_count < 2:
        action = "ABSTAIN"
        confidence = 0.55
        parent = None
    else:
        parent = winner
        action = "KEEP" if declared == winner else ("ADD" if not declared else "REPLACE")
        confidence = min(0.95, 0.65 + 0.05 * top_score)

    supporting = [f"{c['source']}: {c['parent']} ({c['relation']})" for c in independent if c["parent"] == winner]
    conflicting = [f"{c['source']}: {c['parent']}" for c in independent if c["parent"] != winner]
    if declared and declared != winner:
        conflicting.append(f"declared metadata: {declared}")

    return {
        "model_id": item["model_id"], "action": action, "proposed_parent": parent,
        "relationship": relations[winner][0] if parent else None,
        "confidence": round(confidence, 2), "supporting_evidence": supporting,
        "conflicting_evidence": conflicting + errors,
        "rationale": (
            "The strongest parent is independently supported by configuration and documentation."
            if action != "ABSTAIN" else
            "The evidence is conflicting or does not meet the minimum support threshold."
        ),
    }


if LANGCHAIN_AVAILABLE:
    audit_chain = RunnableLambda(extract_evidence) | RunnableLambda(decide_lineage)
    def audit(bundle):
        return audit_chain.invoke(bundle)
else:
    def audit(bundle):
        return decide_lineage(extract_evidence(bundle))


## Test results

In [5]:
results = [audit(sample) for sample in SAMPLES]
print(f"{'MODEL':38} {'ACTION':9} {'PARENT':32} CONFIDENCE")
print("-" * 94)
for result in results:
    parent = result["proposed_parent"] or "-"
    print(f"{result['model_id']:38} {result['action']:9} {parent:32} {result['confidence']:.2f}")


MODEL                                  ACTION    PARENT                           CONFIDENCE
----------------------------------------------------------------------------------------------
demo/missing-lineage-model             ADD       meta-llama/Llama-3.2-1B          0.90
demo/conflicting-lineage-model         REPLACE   meta-llama/Llama-3.2-1B          0.90
demo/correct-lineage-model             KEEP      meta-llama/Llama-3.2-1B          0.90
demo/ambiguous-lineage-model           ABSTAIN   -                                0.45
demo/tool-failure-model                ABSTAIN   -                                0.10


## Evidence-backed structured output example

In [6]:
print(json.dumps(results[1], indent=2))


{
  "model_id": "demo/conflicting-lineage-model",
  "action": "REPLACE",
  "proposed_parent": "meta-llama/Llama-3.2-1B",
  "relationship": "fine-tune",
  "confidence": 0.9,
  "supporting_evidence": [
    "config.json: meta-llama/Llama-3.2-1B (fine-tune)",
    "README: meta-llama/Llama-3.2-1B (fine-tune)"
  ],
  "conflicting_evidence": [
    "declared metadata: mistralai/Mistral-7B-v0.1"
  ],
  "rationale": "The strongest parent is independently supported by configuration and documentation."
}


## Phase 3 work remaining

- Connect repository retrieval to selected public Hugging Face model IDs.
- Add an LLM-backed prose extractor using the prepared prompt and JSON schema.
- Freeze evidence bundles for final tests and record citations.
- Compare decisions with manually verified labels and document one failure case.